# Local Neutralisation Profiles — Radial and Axial

Core-averaged density profiles and eta(z) from 3-D WarpX field arrays.

> Global ParticleNumber ratios are necessary but **not sufficient** to claim
> local space-charge compensation. This notebook provides spatial evidence.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RESULTS_DIR    = _ROOT / 'results'
RUNS_DIR       = _ROOT / 'results'
PLOTS_DIR      = _ROOT / 'plots'
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
_DEFAULTS = {
    'plasma cell z-range [m]':  '0.00 - 0.20',
    'beam-core radius [mm]':    2.0,
    'r_max for profiles [mm]':  15.0,
    'radial bins':              60,
    'grid (synthetic)':         '31 x 31 x 50',
    'eta_target (synthetic)':   0.70,
}
print_simulation_config(
    notebook_title='Local Neutralisation Profiles',
    defaults=_DEFAULTS, overrides={},
)


## 1. 3-D density arrays (synthetic demo)

Replace `ne_3d, ni_3d, np_3d, x, y, z` with arrays from a WarpX plotfile
(e.g. via `yt`) for production use.


In [ ]:
from plasma_column.plotting import (
    setup_publication_style, plot_radial_density_profile, plot_neutralization_vs_z,
)
from plasma_column.diagnostics import (
    compute_radial_density_profiles,
    compute_local_neutralization_vs_z,
    compute_local_core_neutralization,
)
from plasma_column._testing import generate_synthetic_3d_grid
setup_publication_style()

ETA_TARGET = 0.70
R_CORE_M   = 0.002

ne_3d, ni_3d, np_3d, x, y, z = generate_synthetic_3d_grid(
    nx=31, ny=31, nz=50, n_proton_peak=1e15, eta_target=ETA_TARGET,
)
print('Grid shape:', ne_3d.shape, '  z:', f'{z[0]:.3f} - {z[-1]:.3f} m')


## 2. Radial density profiles


In [ ]:
radial_df = compute_radial_density_profiles(
    ne_3d, ni_3d, np_3d, x, y, z,
    z_min_col=0.0, z_max_col=0.20, r_max=0.015, n_bins=60,
)
p, _ = plot_radial_density_profile(
    radial_df, PLOTS_DIR,
    case_name='local_profiles_demo', highlight_core_r=R_CORE_M,
)
plt.show()
print('Saved:', p.name)
display(radial_df.head())


## 3. Axial neutralisation profile eta(z)


In [ ]:
z_df = compute_local_neutralization_vs_z(
    ne_3d, ni_3d, np_3d, x, y, z, r_core=R_CORE_M,
)
p, _ = plot_neutralization_vs_z(
    z_df, PLOTS_DIR, case_name='local_profiles_demo', z_col_range=(0.0, 0.20),
)
plt.show()
print('Saved:', p.name)
display(z_df[['z','eta_electron_only_local_z','eta_net_local_z','keff_over_k0_local_z']].head(10))


## 4. Core-volume summary


In [ ]:
core = compute_local_core_neutralization(
    ne_3d, ni_3d, np_3d, x, y, z,
    z_min_col=0.0, z_max_col=0.20, r_core=R_CORE_M,
)
print('Core-averaged diagnostics:')
for k, v in core.items():
    print(f'  {k:<38} {v:.4g}')
